# 04 · Remotes, GitHub Workflow, Stash & Tags

**Module 16 – Git & Version Control** · Estimated time: 50 min

> From 📘 *Essential Remote Repository Commands* and *Git Stash*, run against a **local "fake GitHub"** (a bare repository) so you can see real push/pull output. Adds authentication, pull requests, tags, and ⚠️ the deprecated `git stash save`.

### 🎯 What you will learn
- Remotes: `remote -v`, `remote add`, `set-url`, `clone`
- `push` (with upstream `-u`), `fetch`, `pull`, remote branches
- Authenticating to GitHub (SSH keys / tokens / `gh`), since passwords no longer work
- The fork → branch → pull request workflow
- `git stash` in depth
- Tagging releases

In [1]:
import os, tempfile
PLAYGROUND = tempfile.mkdtemp(prefix="git-playground-")
os.environ.update({
    "PLAYGROUND": PLAYGROUND, "GIT_CONFIG_GLOBAL": os.devnull,
    "GIT_AUTHOR_NAME": "Demo Learner", "GIT_AUTHOR_EMAIL": "learner@example.com",
    "GIT_COMMITTER_NAME": "Demo Learner", "GIT_COMMITTER_EMAIL": "learner@example.com",
    "GIT_PAGER": "cat", "GIT_EDITOR": "true", "GIT_MERGE_AUTOEDIT": "no",
})

## 1. Set up a "remote"

On GitHub you'd click *New repository*. Here we simulate the server with a **bare repository** (a repo with no working files, which is exactly what servers store):

In [2]:
%%bash
cd "$PLAYGROUND"
git init -q --bare -b main server.git           # our pretend GitHub
git init -q -b main alice && cd alice           # Alice's local project
echo "# Team Project" > README.md
git add . && git commit -q -m "Initial commit"
echo "local repo ready"

local repo ready


## 2. Connecting to remotes (📘)

In [3]:
%%bash
cd "$PLAYGROUND/alice"
git remote add origin "$PLAYGROUND/server.git"   # on GitHub: git remote add origin https://github.com/username/repository.git
git remote -v                                     # view remotes ("origin" is the conventional name)

origin	/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/git-playground-6ptoy0sh/server.git (fetch)
o

rigin	/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/git-playground-6ptoy0sh/server.git (push)


To change a remote's URL (e.g. after renaming the repo on GitHub, or switching from HTTPS to SSH):
```bash
git remote set-url origin git@github.com:username/new-repo.git
```

## 3. Push

In [4]:
%%bash
cd "$PLAYGROUND/alice"
git push -u origin main 2>&1        # first push: -u sets "upstream" so later you can just type `git push`

To /var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/git-playground-6ptoy0sh/server.git
 * [new branc

h]      main -> main


branch 'main' set up to track 'origin/main'.


In [5]:
%%bash
cd "$PLAYGROUND/alice"
echo "Alice's notes" > notes.md
git add . && git commit -q -m "Add notes"
git push 2>&1                       # regular push
git status -sb                      # "## main...origin/main" = in sync

To /var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/git-playground-6ptoy0sh/server.git
   c1e6184..b

bc4843  main -> main


## main...origin/main


## 4. Clone, fetch and pull

Bob clones the project:

In [6]:
%%bash
cd "$PLAYGROUND"
git clone -q server.git bob 2>&1
cd bob && git log --oneline && git remote -v

bbc4843 Add notes
c1e6184 Initial commit


origin	/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/git-playground-6ptoy0sh/server.git (fetch)
o

rigin	/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/git-playground-6ptoy0sh/server.git (push)


Alice pushes a new commit; Bob gets it:

In [7]:
%%bash
cd "$PLAYGROUND/alice"
echo "v2 of notes" >> notes.md && git commit -q -am "Update notes" && git push -q 2>&1

cd "$PLAYGROUND/bob"
git fetch 2>&1                      # download new commits, but DON'T change your files yet
git status -sb                      # "behind 1"
git log --oneline main..origin/main # what's new on the remote
git pull 2>&1                       # fetch + merge (git pull --rebase to rebase instead)
git log --oneline -3

From /var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/git-playground-6ptoy0sh/server
   bbc4843..66d

8658  main       -> origin/main


## main...origin/main [behind 1]


66d8658 Update notes


Updating bbc4843..66d8658
Fast-forward


 notes.md | 1 +
 1 file changed, 1 insertion(+)


66d8658 Update notes


bbc4843 Add notes
c1e6184 Initial commit


| Command | Effect |
|---|---|
| `git fetch` | download remote changes into `origin/main` etc.; your branch is untouched (safe to inspect) |
| `git fetch --all` | fetch from all remotes |
| `git pull` | `fetch` + `merge` into the current branch |
| `git pull origin branch-name` | pull a specific branch |

### Remote branches

In [8]:
%%bash
cd "$PLAYGROUND/bob"
git switch -q -c feature/login
echo "login" > login.py && git add . && git commit -q -m "Add login"
git push -u origin feature/login 2>&1 | tail -2      # push a specific branch
git branch -r                                        # list remote-tracking branches
git push origin --delete feature/login 2>&1 | tail -1   # delete a remote branch
git switch -q main

 * [new branch]      feature/login -> feature/login
branch 'feature/login' set up to track 'origin/f

eature/login'.


  origin/HEAD -> origin/main
  origin/feature/login
  origin/main


 - [deleted]         feature/login


### What if the remote moved on? (the classic "rejected" push)

In [9]:
%%bash
cd "$PLAYGROUND/alice"
echo "alice edit" > a.txt && git add . && git commit -q -m "Alice: add a.txt" && git push -q 2>&1
cd "$PLAYGROUND/bob"
echo "bob edit" > b.txt && git add . && git commit -q -m "Bob: add b.txt"
git push 2>&1 | head -4 || true
echo "---- fix: pull first, then push ----"
git pull --no-rebase -q 2>&1
git push -q 2>&1 && git log --oneline --graph -4

To /var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/git-playground-6ptoy0sh/server.git
 ! [rejected]

        main -> main (fetch first)
error: failed to push some refs to '/var/folders/96/f6xk04z91_gdy

stl3qfqv33r0000gn/T/git-playground-6ptoy0sh/server.git'
hint: Updates were rejected because the remo

te contains work that you do not


---- fix: pull first, then push ----


*   60b0898 Merge branch 'main' of /var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/git-playground-6

ptoy0sh/server
|\  


| * b94f5fa Alice: add a.txt


* | 98b1230 Bob: add b.txt
|/  
* 66d8658 Update notes


## 5. Authenticating with GitHub

> ⚠️ **Changed in 2021:** GitHub **no longer accepts your account password** for `git push`/`pull` over HTTPS. Use one of:
> 1. **GitHub CLI**: `gh auth login` (easiest; sets everything up)
> 2. **SSH keys**:
>    ```bash
>    ssh-keygen -t ed25519 -C "you@example.com"     # create a key pair
>    cat ~/.ssh/id_ed25519.pub                       # paste into GitHub → Settings → SSH keys
>    git remote set-url origin git@github.com:username/repo.git
>    ```
> 3. **Personal Access Token** (fine-grained) used instead of a password, stored by **Git Credential Manager**

## 6. The GitHub collaboration workflow

1. **Fork** the repository (for projects you don't own) or get write access
2. `git clone` your fork, then `git switch -c feature/my-change`
3. Commit your work, `git push -u origin feature/my-change`
4. Open a **Pull Request (PR)** on GitHub: teammates review, CI runs tests
5. Address feedback with more commits; then **merge** (merge commit, squash or rebase)
6. Delete the branch; `git switch main && git pull`

To stay in sync with the original project when you work on a fork:
```bash
git remote add upstream https://github.com/original-owner/repo.git
git fetch upstream && git merge upstream/main
```

## 7. Git stash (📘)

`git stash` **temporarily shelves** uncommitted changes so you can switch tasks, then re-apply them later.

In [10]:
%%bash
cd "$PLAYGROUND/alice"
git pull -q --no-rebase 2>&1
echo "half-finished feature" >> notes.md
echo "scratch" > scratch.txt              # untracked
git status --short
git stash push -m "work in progress on feature X"    # ✅ stash tracked changes with a message
git status --short                        # notes.md clean again (scratch.txt is untracked, so not stashed)
git stash push -u -m "including untracked files"     # -u also stashes untracked files
git status --short
git stash list

 M notes.md
?? scratch.txt


Saved working directory and index state On main: work in progress on feature X


?? scratch.txt


Saved working directory and index state On main: including untracked files


stash@{0}: On main: including untracked files


stash@{1}: On main: work in progress on feature X


> ⚠️ **Deprecated (📘 uses it):** `git stash save "message"`. Use **`git stash push -m "message"`** instead; `push` also accepts file paths (`git stash push -m "msg" file1.py`).

In [11]:
%%bash
cd "$PLAYGROUND/alice"
git stash show stash@{1}                  # files in a specific stash
git stash show -p stash@{1}               # the full diff
git stash apply stash@{1}                 # apply WITHOUT removing it from the list
git status --short
git stash list

 notes.md | 1 +
 1 file changed, 1 insertion(+)


diff --git a/notes.md b/notes.md
index 29aef5f..0e9b114 100644
--- a/notes.md
+++ b/notes.md
@@ -1,2

 +1,3 @@
 Alice's notes
 v2 of notes
+half-finished feature


On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use 

"git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard chan

ges in working directory)
	modified:   notes.md

no changes added to commit (use "git add" and/or "g

it commit -a")


 M notes.md


stash@{0}: On main: including untracked files


stash@{1}: On main: work in progress on feature X


In [12]:
%%bash
cd "$PLAYGROUND/alice"
git restore notes.md
git stash pop                             # apply the most recent stash AND remove it
git stash list
git stash drop stash@{0}                  # remove a specific stash
git stash list
git stash clear                           # remove ALL stashes
echo "stashes left: $(git stash list | wc -l)"

Already up to date.


On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file

>..." to include in what will be committed)
	scratch.txt

nothing added to commit but untracked file

s present (use "git add" to track)


Dropped refs/stash@{0} (5d8106e7c1e392c3ff01b8f63f9c9cba02ef31b0)


stash@{0}: On main: work in progress on feature X


Dropped stash@{0} (06883b7ca7a4583c0a6e47068e522e9ba335f58c)


stashes left:        0


### Common stash use cases (📘)

```bash
# Urgent bug on main while you're mid-feature
git stash
git switch main
# ... fix the bug, commit ...
git switch feature-branch
git stash pop

# Pull when you have local uncommitted changes
git stash
git pull
git stash pop

# Turn a stash into a branch
git stash branch new-feature-branch

# Stash only some hunks (interactive)
git stash -p
```

### Stash command summary

| Command | Description |
|---|---|
| `git stash` / `git stash push -m "msg"` | save changes to the stash |
| `git stash -u` | include untracked files |
| `git stash list` | list stashes |
| `git stash show [-p] [stash@{n}]` | view contents |
| `git stash apply [stash@{n}]` | apply, keep in list |
| `git stash pop` | apply and remove |
| `git stash drop [stash@{n}]` | delete one |
| `git stash clear` | delete all |
| `git stash branch <name>` | create a branch from a stash |

## 8. Tags: marking releases

In [13]:
%%bash
cd "$PLAYGROUND/alice"
git tag -a v1.0.0 -m "First stable release"      # annotated tag (recommended for releases)
git tag v1.0.1                                   # lightweight tag
git tag
git push origin --tags 2>&1 | tail -2
git show v1.0.0 --stat --oneline | head -5

v1.0.0
v1.0.1


 * [new tag]         v1.0.0 -> v1.0.0
 * [new tag]         v1.0.1 -> v1.0.1


tag v1.0.0

First stable release
60b0898 Merge branch 'main' of /var/folders/96/f6xk04z91_gdystl3qfq

v33r0000gn/T/git-playground-6ptoy0sh/server



Use **semantic versioning**: `MAJOR.MINOR.PATCH` (e.g. `2.4.1`): major = breaking changes, minor = new features, patch = bug fixes. On GitHub, tags become **Releases**.

## 📝 Summary
- `git remote add origin URL`, `git push -u origin main`, then `git push` / `git pull`.
- `fetch` downloads; `pull` = fetch + merge; resolve "rejected" pushes by pulling first.
- Authenticate to GitHub with `gh auth login`, SSH keys or tokens, not passwords.
- Collaborate via branches + Pull Requests.
- `git stash push -m` (⚠️ not `stash save`), `list`, `apply`, `pop`, `drop`, `clear`.
- Tag releases with `git tag -a vX.Y.Z`.

## 🏋️ Practice
1. Create a repository on GitHub, connect your local `git-practice` repo with `git remote add origin ...`, and push it.
2. Clone it into a second folder, make a commit there, push, then `git pull` in the first folder.
3. Start editing a file, stash the change with a message, switch branches, come back and `git stash pop`.
4. Tag your current commit as `v0.1.0` and push the tag.